# Black-box and adaptive attacks against the detector
Runtime > Run all. Allow Google Drive access when asked. For every dataset and seed it trains the RF and XGBoost models on clean flows, then attacks original malware flows with three random-search attackers (A: knows only the RF, B: knows RF and XGBoost, C: also knows the detector and its threshold) and reports how many attacked flows evade the IDS and how many of them the ensemble detector still flags.

In [ ]:
import os, time, pickle, warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
except ImportError:
    drive = None                      # not running in Colab
if drive is not None:
    drive.mount("/content/drive")     # errors are shown, not hidden
    assert os.path.exists("/content/drive/MyDrive"), "Google Drive did not mount - re-run and click Allow"

# ---------------- CONFIG (edit here) ----------------
BASE = os.environ.get("DRELAB_BASE", "/content/drive/MyDrive/ColabNotebooks/DReLAB")
OUT = os.environ.get("REV_OUT", "/content/drive/MyDrive/ColabNotebooks/revision_results")
FAMILIES = {"CTU": ["Virut", "Neris", "Rbot", "Menti", "Murlo"],
            "BOTNET": ["Virut", "Neris", "Rbot", "Menti", "Murlo"],
            "CICIDS2018": ["Zeus_Ares"]}
ATTACKS = [("RF", "DDQN"), ("RF", "SARSA"), ("WnD", "DDQN"), ("WnD", "SARSA")]
SEEDS = list(range(int(os.environ.get("N_SEEDS", "5"))))
N_CHUNKS = int(os.environ.get("N_CHUNKS", "20"))       # position blocks
BUFFER = 0.02                                          # drop rows within 2% of a chunk edge
MAX_BENIGN = int(os.environ.get("MAX_BENIGN", "150000"))        # systematic (ordered) subsample of benign
MAX_TRAIN_PER_CLASS = int(os.environ.get("MAX_TRAIN", "40000"))
MAX_EVAL_PER_SET = int(os.environ.get("MAX_EVAL", "30000"))     # cap per val/test set (speed)
FAR_TARGET_PCT = 95                                    # threshold = 95th percentile of clean validation scores
BITS = 8                                               # quantization bits for the cleaned-up squeezer
DROP = {"Unnamed: 0", "Label", "label", "n_actions"}
os.makedirs(OUT, exist_ok=True)


In [ ]:
# ---------------- DATA ----------------
def with_pos(df):
    df = df.copy()
    df["_pos"] = (np.arange(len(df)) + 0.5) / len(df)
    return df

def load_dataset(ds):
    d = f"{BASE}/{ds}"
    benign = pd.read_csv(f"{d}/benign.csv")
    if len(benign) > MAX_BENIGN:                       # ordered subsample, keeps temporal structure
        benign = benign.iloc[np.linspace(0, len(benign) - 1, MAX_BENIGN).astype(int)]
    fams = FAMILIES[ds]
    orig = [with_pos(pd.read_csv(f"{d}/malicious/{f}.csv")) for f in fams]
    adv = {(o, a): [with_pos(pd.read_csv(f"{d}/{o}/{a}/{f}.csv")) for f in fams] for o, a in ATTACKS}
    benign = with_pos(benign)
    ref = [benign] + orig + [x for v in adv.values() for x in v]
    feats = [c for c in benign.columns
             if c not in DROP and c != "_pos"
             and all(c in r.columns for r in ref)
             and all(pd.api.types.is_numeric_dtype(r[c]) for r in ref)]
    def pool(frames):
        df = pd.concat(frames, ignore_index=True)
        X = df[feats].replace([np.inf, -np.inf], np.nan)
        X = X.fillna(X.median()).values.astype(np.float32)
        return X, df["_pos"].values
    data = {"benign": pool([benign]), "orig": pool(orig)}
    for k, v in adv.items():
        data[k] = pool(v)
    return data, feats

def split_roles(seed):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(N_CHUNKS)
    n_tr, n_va = int(N_CHUNKS * 0.5), int(N_CHUNKS * 0.2)
    role = np.empty(N_CHUNKS, dtype=object)
    role[perm[:n_tr]] = "train"
    role[perm[n_tr:n_tr + n_va]] = "val"
    role[perm[n_tr + n_va:]] = "test"
    return role

def take(pool, role, split, rng, cap):
    X, pos = pool
    ch = np.minimum((pos * N_CHUNKS).astype(int), N_CHUNKS - 1)
    frac = pos * N_CHUNKS - ch
    m = (role[ch] == split) & (frac > BUFFER) & (frac < 1 - BUFFER)
    Xs = X[m]
    if len(Xs) > cap:
        Xs = Xs[rng.choice(len(Xs), cap, replace=False)]
    return Xs


In [ ]:
# ---------------- SQUEEZERS ----------------
def make_quant(Xtr, bits=BITS, q=(0.5, 99.5)):
    """True k-bit quantization: clip to the training [q_lo, q_hi] range, then 2^k levels per feature."""
    lo = np.percentile(Xtr, q[0], axis=0); hi = np.percentile(Xtr, q[1], axis=0)
    span = np.maximum(hi - lo, 1e-8); L = 2 ** bits - 1
    def sq(X):
        Z = np.clip((X - lo) / span, 0, 1)
        return (np.round(Z * L) / L * span + lo).astype(np.float32)
    return sq

def paper_sq(X):
    """The original paper code: round(x*256)/256, then a trailing median over 3 adjacent feature columns."""
    X1 = np.round(X * 256) / 256
    s1 = np.full_like(X1, np.nan); s1[:, 1:] = X1[:, :-1]
    s2 = np.full_like(X1, np.nan); s2[:, 2:] = X1[:, :-2]
    return np.nanmedian(np.stack([X1, s1, s2]), axis=0).astype(np.float32)

# ---------------- SCORES ----------------
def conf(p): return np.maximum(p, 1 - p)

def base_probs(X, rf, xgb):
    return rf.predict_proba(X)[:, 1], xgb.predict_proba(X)[:, 1]

def independent_scores(X, rf, xgb, rng):
    """Detectors that do not depend on the squeezer."""
    p_rf, p_xg = base_probs(X, rf, xgb)
    pn = rf.predict_proba(X + rng.normal(0, 0.01, X.shape).astype(np.float32))[:, 1]
    return {"Noise sensitivity (RF)": np.abs(p_rf - pn),
            "Ensemble disagreement only": np.abs(p_rf - p_xg)}

def squeeze_scores(X, rf, xgb, sq):
    p_rf, p_xg = base_probs(X, rf, xgb)
    Xs = sq(X)
    s_rf, s_xg = base_probs(Xs, rf, xgb)
    return {"Feature squeezing (RF)": np.abs(p_rf - s_rf),
            "Confidence drop (RF)": np.abs(conf(p_rf) - conf(s_rf)),
            "Squeeze + ensemble, Eq.11 variance": np.var(np.vstack([s_rf, s_xg]), axis=0),
            "Squeeze + ensemble, Eq.24 sum": np.abs(p_rf - s_rf) + np.abs(p_xg - s_xg)}

PROPOSED = ["Squeeze + ensemble, Eq.11 variance", "Squeeze + ensemble, Eq.24 sum"]
BASELINES = ["Noise sensitivity (RF)", "Feature squeezing (RF)", "Confidence drop (RF)"]

# ---------------- METRICS ----------------
def metrics(neg_val, neg_te, pos_te):
    thr = np.percentile(neg_val, FAR_TARGET_PCT)
    tp = int((pos_te > thr).sum()); fn = len(pos_te) - tp
    fp = int((neg_te > thr).sum()); tn = len(neg_te) - fp
    tpr, fpr = tp / len(pos_te), fp / len(neg_te)
    prec = tpr / (tpr + fpr + 1e-12)                  # precision at balanced prevalence
    f1 = 2 * prec * tpr / (prec + tpr + 1e-12)
    y = np.r_[np.zeros(len(neg_te)), np.ones(len(pos_te))]; s = np.r_[neg_te, pos_te]
    return dict(thr=thr, TPR=tpr, FPR=fpr, Precision_bal=prec, F1_bal=f1,
                AUROC=roc_auc_score(y, s), AUPRC=average_precision_score(y, s),
                TP=tp, FN=fn, FP=fp, TN=tn)


In [ ]:
# ---- BLACK-BOX AND ADAPTIVE ATTACKS: how does the ensemble detector behave when the attacker is smarter than DReLAB? ----
# Attacker: random search with score (probability) access, volume/timing features only (as in DReLAB), derived features kept consistent.
from sklearn.metrics import roc_auc_score
N_ATTACK, ITERS, SIGMA, FMAX = 2000, 60, 0.5, 10.0       # flows per run, queries per flow, search step, max scale factor
CHECKPOINTS = (10, 30, 60)
ATTACKS_BB = {"A: black-box vs RF only": "A", "B: black-box vs RF and XGBoost": "B", "C: adaptive (knows detector and threshold)": "C"}

def split_sets(D, seed):          # identical sampling to the earlier runs (all features)
    role = split_roles(seed); rng = np.random.RandomState(seed)
    g = lambda k, sp, cap: take(D[k], role, sp, rng, cap)
    Xb, Xo = g("benign", "train", 10**9), g("orig", "train", 10**9)
    n = min(len(Xb), len(Xo), MAX_TRAIN_PER_CLASS)
    Xb, Xo = Xb[rng.choice(len(Xb), n, replace=False)], Xo[rng.choice(len(Xo), n, replace=False)]
    return dict(train=(np.vstack([Xb, Xo]), np.r_[np.zeros(n), np.ones(n)]),
                val=np.vstack([g("benign", "val", MAX_EVAL_PER_SET), g("orig", "val", MAX_EVAL_PER_SET)]),
                neg=np.vstack([g("benign", "test", MAX_EVAL_PER_SET), g("orig", "test", MAX_EVAL_PER_SET)]))

NEED = ["Dur", "InBytes", "OutBytes", "TotPkts", "TotBytes", "BytesPerPkt", "BytesPerSec", "PktsPerSec", "RatioOutIn"]
def perturb(X0, F, ix):
    """Scale duration, bytes in, bytes out and packet count by factors >= 1 (padding / delay); keep derived features consistent."""
    X = X0.copy()
    dur0, in0, out0, pk0, tot0 = (X0[:, ix[c]] for c in ("Dur", "InBytes", "OutBytes", "TotPkts", "TotBytes"))
    dur = dur0 * F[:, 0]; inb = np.round(in0 * F[:, 1]); outb = np.round(out0 * F[:, 2]); pk = np.maximum(np.round(pk0 * F[:, 3]), 1)
    tot = tot0 + (inb - in0) + (outb - out0)
    ratio = lambda a, b: np.where(b > 0, a / np.maximum(b, 1e-12), 1.0)
    kd, kt, kp, ko, ki = ratio(dur, dur0), ratio(tot, tot0), ratio(pk, pk0), ratio(outb, out0), ratio(inb, in0)
    X[:, ix["Dur"]] = dur; X[:, ix["InBytes"]] = inb; X[:, ix["OutBytes"]] = outb; X[:, ix["TotPkts"]] = pk; X[:, ix["TotBytes"]] = tot
    X[:, ix["BytesPerPkt"]] = X0[:, ix["BytesPerPkt"]] * kt / np.maximum(kp, 1e-12)
    X[:, ix["BytesPerSec"]] = X0[:, ix["BytesPerSec"]] * kt / np.maximum(kd, 1e-12)
    X[:, ix["PktsPerSec"]] = X0[:, ix["PktsPerSec"]] * kp / np.maximum(kd, 1e-12)
    X[:, ix["RatioOutIn"]] = X0[:, ix["RatioOutIn"]] * ko / np.maximum(ki, 1e-12)
    return X

def run_attack(kind, X0, ix, T, rf, xg, tau, rng):
    n = len(X0); F = np.ones((n, 4))
    obj = {"A": lambda pr, px: pr, "B": lambda pr, px: np.maximum(pr, px),
           "C": lambda pr, px: np.maximum(pr, px) + 5 * np.maximum(0, np.abs(pr - px) - tau)}[kind]
    Z0 = T(X0); pr, px = rf.predict_proba(Z0)[:, 1], xg.predict_proba(Z0)[:, 1]; cur = obj(pr, px); snaps = {0: (pr.copy(), px.copy())}
    for it in range(1, ITERS + 1):
        Fc = np.clip(F * np.exp(rng.normal(0, SIGMA, F.shape)), 1, FMAX)
        Zn = T(perturb(X0, Fc, ix)); prn, pxn = rf.predict_proba(Zn)[:, 1], xg.predict_proba(Zn)[:, 1]
        new = obj(prn, pxn); b = new < cur
        F[b] = Fc[b]; pr[b] = prn[b]; px[b] = pxn[b]; cur[b] = new[b]
        if it in CHECKPOINTS: snaps[it] = (pr.copy(), px.copy())
    return snaps

rows = []
for ds in FAMILIES:
    print("loading", ds)
    D, feats = load_dataset(ds); ix = {c: feats.index(c) for c in NEED}
    for seed in SEEDS:
        t0 = time.time()
        S = split_sets(D, seed); Xtr, ytr = S["train"]
        sc = StandardScaler().fit(Xtr); T = lambda X, sc=sc: sc.transform(X).astype(np.float32)
        Z = T(Xtr)
        rf = RandomForestClassifier(150, max_depth=20, n_jobs=-1, random_state=seed).fit(Z, ytr)
        xg = XGBClassifier(n_estimators=150, max_depth=8, learning_rate=0.1, eval_metric="logloss", random_state=seed).fit(Z, ytr)
        ens = lambda Zx: np.abs(rf.predict_proba(Zx)[:, 1] - xg.predict_proba(Zx)[:, 1])
        tau = np.percentile(ens(T(S["val"])), FAR_TARGET_PCT); s_neg = ens(T(S["neg"]))
        role = split_roles(seed)
        X_all = take(D["orig"], role, "test", np.random.RandomState(1000 + seed), 5 * N_ATTACK)       # original malware, test partition
        mal = rf.predict_proba(T(X_all))[:, 1] >= 0.5                                                      # only flows the IDS currently catches
        X0 = X_all[mal][:N_ATTACK]
        if len(X0) < 50: print("  too few attackable flows", ds, seed); continue
        for name, kind in ATTACKS_BB.items():
            snaps = run_attack(kind, X0, ix, T, rf, xg, tau, np.random.RandomState(seed))
            for q, (pr, px) in snaps.items():
                D_ = np.abs(pr - px); ev_rf = pr < 0.5; ev_both = ev_rf & (px < 0.5); fl = D_ > tau
                auroc = roc_auc_score(np.r_[np.zeros(len(s_neg)), np.ones(len(D_))], np.r_[s_neg, D_]) if q == ITERS else np.nan
                rows.append(dict(dataset=ds, seed=seed, attack=name, queries=q, n=len(X0), evade_RF=ev_rf.mean(), evade_both=ev_both.mean(),
                                 flagged_all=fl.mean(), flagged_among_RF_evaders=(fl[ev_rf].mean() if ev_rf.any() else np.nan),
                                 undetected_RF_evasion=(ev_rf & ~fl).mean(), undetected_both_evasion=(ev_both & ~fl).mean(), AUROC=auroc))
        print(f"  {ds} seed {seed} done in {time.time() - t0:.0f}s ({len(X0)} flows attacked)")
    del D

bb = pd.DataFrame(rows); bb.to_csv(f"{OUT}/blackbox_attack_results.csv", index=False)
pm = lambda x: f"{np.mean(x):.3f}+/-{np.std(x, ddof=1) if len(x) > 1 else 0:.3f}"
cols = ["evade_RF", "evade_both", "flagged_all", "flagged_among_RF_evaders", "undetected_RF_evasion", "undetected_both_evasion", "AUROC"]
summ = bb.groupby(["dataset", "attack", "queries"])[cols].agg(pm).reset_index(); summ.to_csv(f"{OUT}/blackbox_attack_summary.csv", index=False)
pd.set_option("display.width", 260, "display.max_rows", 300, "display.max_colwidth", 45)
print("\n===== After the full query budget (%d queries per flow); mean +/- std over seeds =====" % ITERS)
print(summ[summ.queries == ITERS].drop(columns="queries").to_string(index=False))
print("\n===== Undetected evasion of the IDS (evades the RF and is not flagged) versus the query budget =====")
print(summ[summ.queries > 0][["dataset", "attack", "queries", "undetected_RF_evasion", "undetected_both_evasion"]].to_string(index=False))
print("\nColumns: evade_RF = share of attacked malware the RF now calls benign; evade_both = RF and XGBoost both call it benign;")
print("flagged_all = share the detector flags; undetected_* = evades AND is not flagged (the attacker's real success).")
